# Dataset Statistics

For each dataset: sample counts, column type breakdown, cardinalities, total discrete categories, and sparsity.

In [6]:
from pathlib import Path

import pandas as pd

from origami_jsynth.data import load_jsonl, prepare_dataset
from origami_jsynth.evaluation.flatten import flatten_records
from origami_jsynth.evaluation.type_separation import separate_types
from origami_jsynth.registry import DATASETS

DATA_ROOT = Path("..") / "results"
DATASET_NAMES = list(DATASETS.keys())

In [7]:
def analyze_dataset(name: str) -> dict | None:
    """Load, flatten, type-separate, and compute stats for one dataset."""
    data_dir = DATA_ROOT / name / "data"
    try:
        prepare_dataset(name, data_dir)
    except Exception as e:
        print(f"  SKIP {name}: {e}")
        return None

    train = load_jsonl(data_dir / "train.jsonl")
    test = load_jsonl(data_dir / "test.jsonl")
    all_records = train + test

    # --- Flatten ---
    df_flat = flatten_records(all_records)
    n_rows, n_flat_cols = df_flat.shape

    # Sparsity on the flattened table (before type separation)
    total_cells = df_flat.size
    nan_cells = int(df_flat.isna().sum().sum())
    sparsity = nan_cells / total_cells if total_cells > 0 else 0.0

    # --- Type separation (force=True so every column is separated) ---
    result = separate_types(df_flat, force=True)
    df_sep = result.df

    # Identify typed sub-columns
    cat_cols = [c for c in df_sep.columns if c.endswith(".cat")]
    num_cols = [c for c in df_sep.columns if c.endswith(".num")]
    bool_cols = [c for c in df_sep.columns if c.endswith(".bool")]

    # Cardinality: number of distinct non-NaN values per typed column
    def cardinalities(cols):
        return {c: int(df_sep[c].dropna().nunique()) for c in cols}

    card_cat = cardinalities(cat_cols)
    card_num = cardinalities(num_cols)
    card_bool = cardinalities(bool_cols)

    total_discrete = sum(card_cat.values()) + sum(card_bool.values())

    return {
        "dataset": name,
        "n_train": len(train),
        "n_test": len(test),
        "n_total": len(all_records),
        "n_flat_cols": n_flat_cols,
        "n_cat": len(cat_cols),
        "n_num": len(num_cols),
        "n_bool": len(bool_cols),
        "card_cat": card_cat,
        "card_num": card_num,
        "card_bool": card_bool,
        "total_discrete": total_discrete,
        "sparsity": sparsity,
    }


results = {}
for name in DATASET_NAMES:
    print(f"Processing {name}...")
    r = analyze_dataset(name)
    if r is not None:
        results[name] = r
        print(f"  done — {r['n_total']} records, {r['n_flat_cols']} flat cols")

Processing adult...
Data already prepared at ../results/adult/data
  done — 48842 records, 15 flat cols
Processing diabetes...
Data already prepared at ../results/diabetes/data
  done — 81413 records, 37 flat cols
Processing electric_vehicles...
Data already prepared at ../results/electric_vehicles/data
  done — 210011 records, 18 flat cols
Processing ddxplus...
Data already prepared at ../results/ddxplus/data
  done — 1160131 records, 100 flat cols
Processing github_issues...
Data already prepared at ../results/github_issues/data
  done — 642099 records, 461 flat cols
Processing yelp...
Data already prepared at ../results/yelp/data
  done — 150346 records, 142 flat cols


## Summary Table

In [8]:
rows = []
for r in results.values():
    rows.append(
        {
            "dataset": r["dataset"],
            "n_train": r["n_train"],
            "n_test": r["n_test"],
            "n_total": r["n_total"],
            "flat_cols": r["n_flat_cols"],
            "n_cat": r["n_cat"],
            "n_num": r["n_num"],
            "n_bool": r["n_bool"],
            "total_discrete": r["total_discrete"],
            "sparsity": f"{r['sparsity']:.1%}",
        }
    )

summary = pd.DataFrame(rows).set_index("dataset")
print(summary.to_string())
summary

                   n_train  n_test  n_total  flat_cols  n_cat  n_num  n_bool  total_discrete sparsity
dataset                                                                                              
adult                32561   16281    48842         15      9      6       0             104     0.0%
diabetes             61059   20354    81413         37     24     13       0            2261     0.0%
electric_vehicles   189010   21001   210011         18     13      6       0            5860    11.1%
ddxplus            1025602  134529  1160131        100     50     50       0            6527    67.1%
github_issues       577890   64209   642099        461    330     18     113            7523    93.0%
yelp                135311   15035   150346        142     53      6      73           26812    77.8%


,n_train,n_test,n_total,flat_cols,n_cat,n_num,n_bool,total_discrete,sparsity
dataset,,,,,,,,,
adult,32561,16281,48842,15,9,6,0,104,0.0%
diabetes,61059,20354,81413,37,24,13,0,2261,0.0%
electric_vehicles,189010,21001,210011,18,13,6,0,5860,11.1%
ddxplus,1025602,134529,1160131,100,50,50,0,6527,67.1%
github_issues,577890,64209,642099,461,330,18,113,7523,93.0%
yelp,135311,15035,150346,142,53,6,73,26812,77.8%


## Per-Column Cardinality

In [9]:
for name, r in results.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}  |  {r['n_total']} records  |  sparsity {r['sparsity']:.1%}")
    print(f"{'=' * 60}")

    # Strip the type suffix for display (e.g. "age.num" → "age")
    def strip_suffix(col, suffix):
        return col[: -len(suffix)] if col.endswith(suffix) else col

    sections = [
        ("Categorical", r["card_cat"], ".cat"),
        ("Numeric", r["card_num"], ".num"),
        ("Boolean", r["card_bool"], ".bool"),
    ]
    for label, cards, suffix in sections:
        if not cards:
            continue
        print(f"\n  {label} ({len(cards)} cols, total discrete: {sum(cards.values()) if label != 'Numeric' else '—'}):")
        col_df = pd.DataFrame(
            [(strip_suffix(c, suffix), v) for c, v in sorted(cards.items(), key=lambda x: -x[1])],
            columns=["column", "cardinality"],
        ).set_index("column")
        print(col_df.to_string())


  adult  |  48842 records  |  sparsity 0.0%

  Categorical (9 cols, total discrete: 104):
                cardinality
column                     
native.country           42
education                16
occupation               15
workclass                 9
marital.status            7
relationship              6
race                      5
income                    2
sex                       2

  Numeric (6 cols, total discrete: —):
                cardinality
column                     
fnlwgt                28523
capital.gain            123
capital.loss             99
hours.per.week           96
age                      74
education.num            16

  diabetes  |  81413 records  |  sparsity 0.0%

  Categorical (24 cols, total discrete: 2261):
                     cardinality
column                          
diag_3                       760
diag_2                       731
diag_1                       693
race                           6
A1Cresult                      4
acarbose  